# Maxwell MHM: full-vector declarations and a propagating TM mode

Write electric and magnetic local equations and the global tangential coupling, then assemble, solve and reconstruct. A stationary 3D patch first isolates conventions. The second example is the smooth 2D transverse-magnetic (TM) cavity of Lanteri et al. (2018); its fields vary in space and time, and its spatial and temporal refinements are assessed separately.

Use the locked **`introduction`** Pixi environment: the independent physical Galerkin baseline below executes UFL and DOLFINx. All cells run; unavailable native dependencies are reported by the notebook runner.


In [ ]:
from pathlib import Path
import os
import sys
from pymhm.io.workspace import workspace_from_archive

# Download verified support files; this operation does not execute them.
COMPANION_URL = "https://ipes-lncc.github.io/pymhm/downloads/bc9d71d84f38d8249dcf23ee81ee586983337418af75777cb22d19329899db56/introductory_methods-companion.zip"
COMPANION_SHA256 = "bc9d71d84f38d8249dcf23ee81ee586983337418af75777cb22d19329899db56"
WORKSPACE = Path(
    os.environ.get("PYMHM_WORKSPACE", Path.cwd() / ".pymhm-companions" / COMPANION_SHA256)
)
ROOT = workspace_from_archive(COMPANION_URL, sha256=COMPANION_SHA256, directory=WORKSPACE)
os.environ["PYMHM_WORKSPACE"] = str(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

# Explicitly prepare declared data with the downloaded Python helpers.
from scripts.notebook_reproduction import notebook_workspace

ROOT = notebook_workspace("waves/maxwell/introductory_methods.ipynb", directory=ROOT)
root = ROOT
print("Workspace:", ROOT)

import numpy as np


## Declare the time-discrete mathematical problem

The shared differential matrix is $C=-\mathrm{curl}$, its electric pairing is
$-C^\top$, and $Q$ includes the signed tangential face maps. The canonical trace
is $\lambda=H\times n$. Exterior data satisfy $E_{\mathrm{tan}}-\lambda=g$.
Here $Z$ is the impedance face mass and $b$ contains the prescribed moments.

For an electric kick of duration $\tau$, use its average field as the local
unknown. The initialization uses $\tau=\Delta t/2$; later kicks use
$\tau=\Delta t$.

$$
\begin{aligned}
 w &= \tfrac12(E_{\mathrm{old}}+E_{\mathrm{new}}),\\
 M_e w+\tfrac{\tau}{2}Q\lambda
   &=M_e E_{\mathrm{old}}+\tfrac{\tau}{2}(F-C^\top H),\\
 -\sum_K Q_K^\top w_K+Z\lambda&=-b,\\
 E_{\mathrm{new}}&=2w-E_{\mathrm{old}}.
\end{aligned}
$$

The magnetic stage uses an independent mass problem with no trace coordinates.
It is assembled and reconstructed by the same generic core.

$$
 M_h H_{\mathrm{new}}=M_h H_{\mathrm{old}}+\Delta t\,C E.
$$

In [ ]:
import numpy as np
from pymhm.core.equations import Equation, LocalEquations
from pymhm.meshes.tetrahedron import TetraMesh
from threadpoolctl import threadpool_limits
from examples.tutorial_maxwell_equations import (
    Discretization, ElectricItem, MagneticItem,
    prepare, initialize, advance, l2_errors, compare_state,
)


def electric_local(item: ElectricItem) -> LocalEquations:
    """Declare the local midpoint field and its canonical tangential coupling."""
    local, Eold, H, F, duration = item
    return LocalEquations(
        a=local.electric_mass,
        L=local.electric_mass @ Eold + duration / 2 * (F - local.curl.T @ H),
        b=duration / 2 * local.coupling,
        c=-local.coupling.T,
        dofs=local.trace_dofs,
    )


def magnetic_local(item: MagneticItem) -> LocalEquations:
    """Declare the magnetic mass update with no global trace unknown."""
    local, E, Hold, dt = item
    return LocalEquations(
        a=local.magnetic_mass,
        L=local.magnetic_mass @ Hold + dt * (local.curl @ E),
        b=0, c=0, dofs=[],
    )


def electric_global(data: Discretization, boundary: np.ndarray) -> Equation:
    """Declare exterior impedance and prescribed tangential moments."""
    return Equation(a=data.impedance, L=-boundary)

## Assemble the declared spaces and physical data

`prepare` tabulates the shared curl and mass kernels; it does not select a
physical solver. The three providers above are passed to the integration
functions and consumed by `MultiscaleProblem` at each stage. Volume mass
projections initialize both fields, and the initial electric field is projected
onto the explicitly constrained tangential moments.

In [ ]:
cube = TetraMesh.unit_cube()
mesh = TetraMesh(cube.points, cube.cells[:2])
electric = np.array([1.2, -0.3, 0.7])
magnetic = np.array([0.4, 0.8, -0.2])


def impedance_data(time: float, points: np.ndarray, normals: np.ndarray) -> np.ndarray:
    """Supply the stationary physical E_tan-(H cross n) boundary data."""
    return electric - np.cross(magnetic, normals)


with threadpool_limits(1):
    data = prepare(
        mesh, time_step=0.001, degree=2, local_refinement=2,
        absorbing=1.0, boundary_data=impedance_data, quadrature_order=5,
    )
    state = initialize(
        data, electric, magnetic,
        electric_provider=electric_local, global_provider=electric_global,
    )
print({
    "dt_times_frequency_bound": data.time_step * data.frequency_bound,
    "CFL_upper_limit": 2.0,
    "electric_time": state.electric_time,
    "magnetic_time": state.magnetic_time,
})

## Advance four steps and inspect separate physical checks

The initialization places $E$ at $\Delta t/2$ and $H$ at zero. Each subsequent
step updates $H$ first and then solves the electric midpoint equations. The
reported energy uses the same cross-time convention as this leapfrog scheme.
Tangential moment defects and original electric rows are checked separately
from physical field errors.

In [ ]:
trajectory = [state]
rows = []
with threadpool_limits(1):
    for step in range(4):
        state = advance(
            data, state,
            electric_provider=electric_local,
            magnetic_provider=magnetic_local,
            global_provider=electric_global,
        )
        trajectory.append(state)
        electric_error, magnetic_error = l2_errors(data, state, electric, magnetic)
        assert max(electric_error, magnetic_error) < 1e-9
        assert abs(state.energy_balance_residual) < 1e-10
        rows.append({
            "step": step + 1,
            "electric_time": state.electric_time,
            "magnetic_time": state.magnetic_time,
            "electric_l2": electric_error,
            "magnetic_l2": magnetic_error,
            "tangential_moment_norm": state.constraint_moment_norm,
            "original_electric_residual": state.original_electric_residual,
            "energy_balance_residual": state.energy_balance_residual,
        })
for row in rows:
    print(row)

## Compare with the original coefficient equations

The primary trajectory above uses the declared user providers. This separate
comparison assembles and solves the original A/B/C/D matrix on identical
geometry, coefficients, spaces, boundary data and times, independently of
Schur elimination. Coefficient agreement is checked at initialization and
after every step, together with field norms, staggered times and energy.


In [ ]:
comparisons = []
with threadpool_limits(1):
    references = [initialize(data, electric, magnetic, original=True)]
    for _ in range(4):
        references.append(advance(data, references[-1], original=True))
    for own, reference in zip(trajectory, references, strict=True):
        difference = compare_state(own, reference)
        assert difference["electric_coefficient_linf"] < 2e-12
        assert difference["magnetic_coefficient_linf"] < 2e-12
        assert difference["trace_coefficient_linf"] < 2e-11
        assert difference["energy_difference"] < 2e-14
        assert difference["electric_time_difference"] == 0.0
        assert difference["magnetic_time_difference"] == 0.0
        reference_errors = l2_errors(data, reference, electric, magnetic, order=5)
        assert max(reference_errors) < 1e-9
        assert abs(reference.energy_balance_residual) < 1e-10
        comparisons.append(difference)
for step, comparison in enumerate(comparisons):
    print({"step": step, **comparison})

## Retain the original analytical tutorial controls

The earlier compatibility helper still checks the same four-step patch. A zero
field control exercises homogeneous data separately; its primary equations use
the same providers and trace orientation.

In [ ]:
with threadpool_limits(1):
    zero_data = prepare(mesh, boundary_data=0.0)
    zero_state = initialize(
        zero_data, np.zeros(3), np.zeros(3),
        electric_provider=electric_local, global_provider=electric_global,
    )
    for _ in range(4):
        zero_state = advance(
            zero_data, zero_state,
            electric_provider=electric_local,
            magnetic_provider=magnetic_local,
            global_provider=electric_global,
        )
    assert l2_errors(zero_data, zero_state, np.zeros(3), np.zeros(3)) == (0.0, 0.0)

from examples.tutorial_vector_variants import run_variant
print(run_variant("maxwell-vector-3d"))
print(run_variant("maxwell-vector-3d", homogeneous=True))

## Plot the electric and magnetic vector fields

Use the returned field object's sample method in its executed DG bases. Fine-cell centroids keep independent incident values. Black edges show the actual macro tetrahedra. Field arrows use a common length multiplier; each title states its physical staggered time.


In [ ]:
import matplotlib.pyplot as plt
from itertools import combinations

fig = plt.figure(figsize=(10, 4.8), layout="constrained")
axes = [fig.add_subplot(1, 2, index+1, projection="3d") for index in range(2)]
for cell in range(len(mesh.cells)):
    points, E, H = state.sample(cell, np.full((1, 4), 0.25))
    points, E, H = points.reshape(-1, 3), E.reshape(-1, 3), H.reshape(-1, 3)
    for ax, values, color in zip(axes, (E, H), ("tab:blue", "tab:orange"), strict=True):
        ax.quiver(*points.T, *values.T, length=0.15, normalize=False, color=color)
        vertices = mesh.points[mesh.cells[cell]]
        for i, j in combinations(range(4), 2):
            ax.plot(*vertices[[i, j]].T, color="black", linewidth=0.6)
for ax, name, time in zip(axes, ("Electric field", "Magnetic field"),
                          (state.electric_time, state.magnetic_time), strict=True):
    ax.set(xlabel="x", ylabel="y", zlabel="z", title=f"{name}, t={time:g}",
           xlim=(-0.1,1.1), ylim=(-0.1,1.1), zlim=(-0.1,1.1))
    ax.set_box_aspect((1,1,1))
Path("build/tutorial-methods").mkdir(parents=True, exist_ok=True)
fig.savefig("build/tutorial-methods/maxwell-vector-fields.png", dpi=200, metadata={"Author":"IPES Research Group", "License":"CC BY 4.0 — https://creativecommons.org/licenses/by/4.0/"})
plt.show()


## Propagate a nontrivial 2D TM cavity mode

The full-vector 3D patch checks orientations; its exact constant fields do not establish approximation rates. For the spatial study, use the original smooth TM cavity data from Section 6.2 of [Lanteri et al. (2018)](https://doi.org/10.1137/16M110037X). Here $E=E_z$ is scalar and $H=(H_x,H_y)$ is a physical planar vector. Unit material coefficients, zero current, homogeneous PEC and zero initial magnetic field give

$$
\begin{aligned}
 E(t,x,y)&=\cos(\omega t)\sin(2\pi x)\sin(2\pi y),\\
 H(t,x,y)&=\frac{\sin(\omega t)}{\sqrt2}
 (-\sin(2\pi x)\cos(2\pi y),\cos(2\pi x)\sin(2\pi y)),\\
 \omega&=2\pi\sqrt2.
\end{aligned}
$$

The analytical derivatives satisfy $\partial_tE=\operatorname{curl}H$ and $\partial_tH=-\operatorname{curl}E$ with the displayed conventions. The local and global providers defined above remain the equations: only their supplied scalar/vector mass and curl operators and tangential trace geometry change. Face degree one uses a single local P3 triangle, as in that literature study.


In [ ]:
from pymhm import FaceSpace, SkeletonSpace, TriangleMesh
from pymhm.fem.vector.curl import TangentialTraceSpace


def electric_shape(points):
    """State the independent scalar TM electric initial field."""
    x,y=2*np.pi*points.T
    return (np.sin(x)*np.sin(y))[:,None]


def magnetic_shape(points):
    """State the independently differentiated planar magnetic mode."""
    x,y=2*np.pi*points.T
    return np.column_stack((-np.sin(x)*np.cos(y),np.cos(x)*np.sin(y)))/np.sqrt(2)


macro_tm=TriangleMesh.unit_square(8)
trace_tm=SkeletonSpace(macro_tm,tuple(FaceSpace.uniform(1) for _ in macro_tm.faces))
with threadpool_limits(1):
    tm_data=prepare(macro_tm,time_step=0.0005,degree=3,local_refinement=1,
                    skeleton=TangentialTraceSpace(trace_tm),absorbing=None,
                    quadrature_order=8)
    tm_state=initialize(tm_data,electric_shape,0.0,
                        electric_provider=electric_local,global_provider=electric_global)
    for _ in range(20):
        tm_state=advance(tm_data,tm_state,electric_provider=electric_local,
                         magnetic_provider=magnetic_local,global_provider=electric_global)
    omega_tm=2*np.pi*np.sqrt(2)
    exact_e=lambda points:np.cos(omega_tm*tm_state.electric_time)*electric_shape(points)
    exact_h=lambda points:np.sin(omega_tm*tm_state.magnetic_time)*magnetic_shape(points)
    e_error,h_error=l2_errors(tm_data,tm_state,exact_e,exact_h)
    assert tm_state.original_electric_residual<=1e-10
    assert abs(tm_state.energy_balance_residual)<=1e-10
print({'electric_time':tm_state.electric_time,'magnetic_time':tm_state.magnetic_time,
       'electric_L2':e_error,'magnetic_L2':h_error,
       'original_electric_relative':tm_state.original_electric_residual})


## Plot the propagating fields with independent incident values

Sample every local polynomial independently. No values are averaged across macrofaces. The exact electric field is evaluated at the stored electric time, and the exact magnetic field at its different stored magnetic time. The macro mesh is visible on every panel.


In [ ]:
from examples.introduction.vector import plot_field_panels

reference=TriangleMesh([[0,0],[1,0],[0,1]],[[0,1,2]]).submesh(0,4)
bary=np.column_stack((1-reference.points.sum(axis=1),reference.points))
points=[];triangles=[];electric_samples=[];magnetic_samples=[];offset=0
for index in range(len(macro_tm.cells)):
    x,E,H=tm_state.sample(index,bary)
    points.append(x.reshape(-1,2))
    electric_samples.append(E.reshape(-1))
    magnetic_samples.append(np.linalg.norm(H.reshape(-1,2),axis=1))
    for fine_cell in range(len(x)):
        triangles.append(reference.cells+offset+fine_cell*len(bary))
    offset+=len(x)*len(bary)
points=np.concatenate(points);triangles=np.concatenate(triangles)
electric_samples=np.concatenate(electric_samples)
magnetic_samples=np.concatenate(magnetic_samples)
exact_electric=exact_e(points).ravel()
exact_magnetic=np.linalg.norm(exact_h(points),axis=1)


## Independently assemble the classical physical TM baseline

The same physical reduction satisfies $E_{tt}-\Delta E=0$ with strong homogeneous PEC, initial $E=\sin(2\pi x)\sin(2\pi y)$ and initial $E_t=0$. Define a conforming global $P_3$ space, write its UFL mass/stiffness, mass-project the initial field and advance with the generic Newmark primitive. This independently assembled reference has no MHM trace, local basis or condensed operator.

Recover $H=-\operatorname{rot}\nabla\int_0^tE\,ds$ by trapezoidal integration. Store it at the MHM magnetic time and electric field at the MHM electric time, so both reference panels use the correct staggered physical times. The separate archived reference refinement uses $n=16,32,64$ and independent analytical error integration before calling a numerical field a baseline.


In [ ]:
import basix
import basix.ufl
import ufl
from dolfinx import fem
from pymhm import compile_form
from pymhm.backends.spaces import bind_space, coefficient_map, evaluate_descriptor_data
from pymhm.fem.scalar.triangle import nodal_space
from pymhm.linalg.dynamics import newmark_step
from pymhm.linalg.linear import factorize

classical_mesh=TriangleMesh.unit_square(64)
classical_space=bind_space(classical_mesh,basix.ufl.element(
    'Lagrange','triangle',3,lagrange_variant=basix.LagrangeVariant.equispaced,
))
try:
    e,v=ufl.TrialFunction(classical_space.space),ufl.TestFunction(classical_space.space)
    domain=classical_space.space.mesh
    x=ufl.SpatialCoordinate(domain)
    phi=ufl.sin(2*np.pi*x[0])*ufl.sin(2*np.pi*x[1])
    dx=ufl.Measure('dx',domain=domain,metadata={'quadrature_degree':14})
    mass=compile_form(e*v*dx)
    stiffness=compile_form(ufl.inner(ufl.grad(e),ufl.grad(v))*dx)
    initial_load=compile_form(phi*v*dx)
    _,nodes=nodal_space(classical_mesh,3)
    portable_exterior=np.flatnonzero(np.any(np.isclose(nodes,0)|np.isclose(nodes,1),axis=1))
    fixed=coefficient_map(classical_space)[portable_exterior]
    free=np.setdiff1d(np.arange(classical_space.size),fixed)
    m,k=mass[free][:,free],stiffness[free][:,free]
    reference_dt=0.000125
    steps_e=round(tm_state.electric_time/reference_dt)
    steps_h=round(tm_state.magnetic_time/reference_dt)
    assert abs(steps_e*reference_dt-tm_state.electric_time)<1e-13
    assert abs(steps_h*reference_dt-tm_state.magnetic_time)<1e-13
    with threadpool_limits(1),factorize(m) as mf,factorize(m+reference_dt**2/4*k) as sf:
        reference_e=mf.solve(initial_load[free])
        reference_velocity=np.zeros_like(reference_e)
        integral_e=np.zeros_like(reference_e)
        for step in range(1,steps_e+1):
            old=reference_e
            reference_e,reference_velocity=newmark_step(
                m,k,sf,mf,reference_dt,reference_e,reference_velocity,
                np.zeros_like(reference_e),np.zeros_like(reference_e),
            )
            integral_e+=reference_dt/2*(old+reference_e)
            if step==steps_h:
                magnetic_integral=integral_e.copy()
    e_coefficients=np.zeros(classical_space.size)
    e_coefficients[free]=reference_e
    classical_e=classical_space.evaluate(e_coefficients,points)
    # The executed native descriptor evaluates the same independent basis gradient.
    integral_coefficients=np.zeros(classical_space.size)
    integral_coefficients[free]=magnetic_integral
    descriptor=classical_space.descriptor()
    _,integral_gradient=evaluate_descriptor_data(descriptor,integral_coefficients,points)
    classical_h=np.column_stack((-integral_gradient[:,1],integral_gradient[:,0]))
    classical_h_magnitude=np.linalg.norm(classical_h,axis=1)
    eh=fem.Function(classical_space.space)
    eh.x.array[:]=e_coefficients
    exact_e_ufl=float(np.cos(omega_tm*tm_state.electric_time))*phi
    electric_l2=float(np.sqrt(fem.assemble_scalar(fem.form((eh-exact_e_ufl)**2*dx))))
finally:
    classical_space.close()
print({'independent_classical_mesh':64,'electric_L2':electric_l2,
       'electric_time':steps_e*reference_dt,'magnetic_time':steps_h*reference_dt})


In [ ]:
figure=plot_field_panels(macro_tm,{
    'Analytical electric field':(points,triangles,exact_electric),
    'MHM electric field':(points,triangles,electric_samples),
    'Conforming P3 electric field':(points,triangles,classical_e),
    'Signed electric error':(points,triangles,electric_samples-exact_electric),
    'Analytical magnetic magnitude':(points,triangles,exact_magnetic),
    'MHM magnetic magnitude':(points,triangles,magnetic_samples),
    'Galerkin TM magnetic magnitude':(points,triangles,classical_h_magnitude),
    'Signed magnetic-magnitude error':(points,triangles,magnetic_samples-exact_magnetic),
},figsize=(13,11))
figure.savefig('build/tutorial-methods/maxwell-tm-fields.png',dpi=200,
               metadata={'Author':'IPES Research Group','License':'CC BY 4.0'})
plt.show()


## Compare the literature spatial rates and the time integrator separately

For this smooth 2D TM family, trace degree one targets order two in the combined electric/magnetic $L^2$ maximum-in-time norm and order one in the combined broken $H(\operatorname{curl})$ norm. The electric field can be superconvergent; its higher observed order is not used as the combined-field target. Retain every coarse level on the plot, and compare three successive terminal intervals and the normalized errors $E/H^q$.

The spatial acquisition repeats the continuum error integrals at every staggered sample. Independent higher error quadrature, original electric equations, PEC moments, the conservative CFL bound and modified energy are checked. A finest-mesh half-time-step run assesses temporal contamination. The separate time curve compares leapfrog with the exact constrained semidiscrete ODE; it establishes integration order and does not serve as a classical physical baseline.

For a classical physical reference, the TM reduction gives $\partial_{tt}E-\Delta E=0$ with strong homogeneous PEC. Independently assemble conforming global P3 UFL mass and stiffness forms, advance with Newmark, and recover $H=-\operatorname{curl}\int_0^tE\,ds$. This reference uses its own successively refined global meshes and directly measured errors against the analytical cavity fields. It is a TM scalar Galerkin reference, not a claim of general three-dimensional Nédélec conformity.


In [ ]:
import json
from examples.tutorial_convergence import refinement_series,plot_method_series,asymptotic_summary
record_path=ROOT/'examples/results/tutorial-methods/maxwell-spatial-asymptotic.json'
record=json.loads(record_path.read_text())
rows=[dict(row,**row['max_in_time']) for row in record['rows']]
spatial=refinement_series(
    record_path,rows,'H',
    {'combined L2':'combined_l2','combined broken H(curl)':'combined_hcurl'},
    {'combined L2':2.0,'combined broken H(curl)':1.0},method='maxwell',
    spaces='Original 2D TM cavity, local P3 on one triangle, face P1; maximum over staggered times',
    rate_provenance='Lanteri et al. (2018), Section6.2: combined L2 order2, broken Hcurl order1',
    refinement='macro grid spacing',root=ROOT,
)
record_path=ROOT/'examples/results/tutorial-methods/maxwell-time-current.json'
record=json.loads(record_path.read_text())
temporal=refinement_series(
    record_path,record['rows'],'time_step',
    {'electric L2':'electric_error_l2','magnetic L2':'magnetic_error_l2'},
    {'electric L2':2.0,'magnetic L2':2.0},method='maxwell-time',
    spaces='Fixed complete constrained spatial operator; staggered electric and magnetic times',
    rate_provenance='Second-order leapfrog against the exact constrained semidiscrete evolution',
    refinement='time step',root=ROOT,
)
from IPython.display import SVG, display

plot_directory = Path("build/tutorial-methods")
plot_directory.mkdir(parents=True, exist_ok=True)
for series in (spatial, temporal):
    print(series["method"], asymptotic_summary(series))
    figure = plot_method_series(series)
    figure_path = plot_directory / f"{series['method']}-convergence.svg"
    figure.savefig(
        figure_path,
        metadata={"Creator": "IPES Research Group", "Rights": "CC BY 4.0"},
    )
    display(SVG(filename=str(figure_path)))
    plt.close(figure)


## References

- Lanteri, Paredes, Scheid and Valentin (2018), *The Multiscale Hybrid-Mixed Method for the Maxwell Equations in Heterogeneous Media*, [DOI:10.1137/16M110037X](https://doi.org/10.1137/16M110037X).

The smooth 2D TM cavity data and declared spaces match the cited verification family. The initial two-tetrahedron full-vector 3D patch is a separate analytical consistency check.
